# MultiHop-RAG (COLM 2024) — multi-doc reasoning

**Paper:** [Tang & Yang, arXiv:2401.15391](https://arxiv.org/abs/2401.15391)  
**Dataset:** HF `yixuantt/MultiHopRAG` — 2,556 queries, evidence across 2–4 news docs.

This harness builds a **closed-world mini subset** (gold evidence articles + distractors), runs local methods, and applies the same **dual scoreboard** as the Hotpot autopsy so GraphRAG isn’t punished for prose.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, Image, display
from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

from rag_benchmark import build_multihop_rag_subset
from rag_benchmark.charts import METHOD_LABELS, print_leaderboard
from rag_benchmark.metric_autopsy import (
    enrich_accuracy,
    method_metric_profile,
    question_catalog,
    scenario_dual_leaderboard,
    write_autopsy_artifacts,
)

N_PER_TYPE = 3
RUN_BENCHMARK = False  # True → python scripts/run_multihop_benchmark.py
RESULTS = ROOT / "results_multihop"
RESULTS.mkdir(parents=True, exist_ok=True)

built = build_multihop_rag_subset(project_root=ROOT, n_per_type=N_PER_TYPE)
display(Markdown("### Subset meta"))
display(pd.Series(built["meta"]).to_frame("value"))


## Question catalog


In [ ]:
catalog = pd.read_csv(ROOT / "results" / "multihop_question_catalog.csv")
display(catalog)
qa = pd.read_json(built["qa_path"])
for i, q in qa.iterrows():
    print(f"Q{i+1} [{q.multihop_type}] {q.id}")
    print(f"  {q.question}")
    print(f"  Gold: {q.expected_answer}\n")


## Results + dual scoreboard

If `RUN_BENCHMARK` is False, loads `results_multihop/`. Generative score is the primary multi-hop leaderboard.


In [ ]:
assert (RESULTS / "accuracy_results.csv").exists(), (
    "Run: PYTHONPATH=src python scripts/run_multihop_benchmark.py"
)
acc = enrich_accuracy(pd.read_csv(RESULTS / "accuracy_results.csv"))
write_autopsy_artifacts(
    results_dir=RESULTS,
    qa_path=Path(built["qa_path"]),
    type_key="multihop_type",
    scenario_col="multihop_type",
)

display(Markdown("### Dual scoreboard by MultiHop-RAG question type"))
display(scenario_dual_leaderboard(acc, scenario_col="multihop_type"))

display(Markdown("### Method profile (generative-first)"))
display(method_metric_profile(acc)[
    ["label", "generative", "extractive", "composite", "llm_judge", "contains", "token_f1", "exact_match"]
].round(3))

profile = method_metric_profile(acc).set_index("label")
fig, ax = plt.subplots(figsize=(10, 4.5))
profile[["generative", "extractive", "composite"]].sort_values("generative").plot(kind="barh", ax=ax)
ax.set_xlim(0, 1)
ax.set_title("MultiHop-RAG mini — generative vs extractive vs composite")
plt.tight_layout()
plt.show()

print_leaderboard(RESULTS)
if (RESULTS / "benchmark_dashboard.png").exists():
    display(Image(filename=str(RESULTS / "benchmark_dashboard.png")))


## How to read this vs Hotpot / GraphRAG-Bench

| Suite | What it stresses | Prefer |
|-------|------------------|--------|
| HotpotQA | Wikipedia bridge/comparison spans | Extractive for EM papers; generative for UX |
| **MultiHop-RAG** | News multi-doc inference / temporal / comparison | **Generative** + retrieval recall |
| GraphRAG-Bench | Fact → creative task ladder | Generative on summarize/creative |

If composite says “vector wins multi-hop” but generative says “graph/hybrid wins”, trust generative for GraphRAG product decisions.
